# SpaceLLaVA: Q1–Q4, 100 снимков / 400 запросов

## Goal
Clean RGB-only прогон: 50 лунных + 50 марсианских снимков из `orbital_300_v4`, четыре вопроса каждому. Исправлен только `mars_seg_018`; маски и высоты не менялись. Кандидатные ответы вычислены для 400/400 вопросов, но ещё не приняты как окончательный scoring GT. Этот notebook собирает предсказания, не считает accuracy.

В ZIP нет эталонных ответов, масок или высот. Сохраняются исходные val/test и группы сцен. До просмотра предсказаний зафиксируй окончательные эталоны/правила допуска, чтобы последующая оценка не зависела от ответов модели.

Локально проверены синтаксис и CPU bootstrap. Запуск на GPU Colab здесь не выполнен; реальная совместимость проверяется новым smoke test. Включи GPU в Runtime → Change runtime type. Не запускай параллельно другую модель.

## Setup

Загрузи `outputs/colab/spacellava_q1_q4_v5_clean.zip` в `MyDrive/planetary_benchmark/`. Открой этот `.ipynb` через Colab → File → Upload notebook. Выполняй ячейки сверху вниз. Новый workspace и RUN_ID отделяют эти результаты от прежнего свободного описания и Q5–Q6.

Веса не входят в ZIP. Если прежняя Colab VM сохранилась, ниже можно подключить уже скачанные `weights/` и `external/` без копирования. Если нет — загрузка требует явного `DOWNLOAD_WEIGHTS=True`. Скачивание здесь автоматически не включается.

In [8]:
from pathlib import Path
import hashlib, json, os, signal, stat, subprocess, sys, uuid, zipfile, shutil
from google.colab import drive
drive.mount('/content/drive')

PERSIST = Path('/content/drive/MyDrive/planetary_benchmark')
BUNDLE_ZIP = PERSIST / 'spacellava_q1_q4_v5_clean.zip'
WORKSPACE = Path('/content/planetary_q1_q4_v5')  # Keep this path identical after reconnect for resume.
ENV_DIR = Path('/content/venv_spacellava')
PYTHON_VERSION = '3.12.15'
TORCH_INDEX = 'https://download.pytorch.org/whl/cu128'
RUN_ID = 'spacellava_q1_q4_v5_clean'
SESSION_ID = uuid.uuid4().hex  # New Colab VM/session needs a fresh smoke test.
DOWNLOAD_WEIGHTS = True  # Set True explicitly to download ~28 GB of pinned public weights/code.
WEIGHTS_SOURCE = None  # Optional existing Drive weights/ folder; copied to local WORKSPACE/weights.
RUN_BENCHMARK = True  # Set True only after the smoke test succeeds.
RESUME = False  # True only when this RUN_ID has a saved checkpoint.
CHECKPOINT_EVERY = 10  # A hard disconnect can lose at most this many uncheckpointed responses.
MAX_REQUESTS = None  # Full bundle; a smaller value is a separate fixed protocol/run_id.
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run(['nvidia-smi'], check=True)
print('Bundle exists:', BUNDLE_ZIP.is_file())
print('Local free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 1))
assert BUNDLE_ZIP.is_file(), 'Upload the ZIP to the configured Drive path first.'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bundle exists: True
Local free GiB: 35.3


### 1. Проверить и распаковать переносимый пакет
Используй только пакет из своего проекта: в нём исполняемый код. Проверяются пути и SHA-256 всех файлов. Повторная ячейка не заменяет изменённые файлы. При новом пакете выбери новый workspace/run, не редактируй старые результаты.

In [2]:
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    info = archive.infolist()
    names = [item.filename for item in info]
    assert len(names) == len(set(names)), 'Duplicate archive entry'
    record = json.loads(archive.read('colab_bundle.json'))
    claimed = record.pop('bundle_sha256')
    digest = hashlib.sha256(json.dumps(record, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()
    assert record['schema'] == 'planetary-colab-v1' and claimed == digest, 'Metadata changed'
    assert set(names) == set(record['files_sha256']) | {'colab_bundle.json'}, 'Unexpected archive entry'
    for item in info:
        relative = Path(item.filename)
        assert not relative.is_absolute() and '..' not in relative.parts
        assert chr(92) not in item.filename and ':' not in item.filename
        assert not any(ord(c) < 32 for c in item.filename)
        assert not set(p.casefold() for p in relative.parts) & {'data_rover', '.env', '.git', 'weights', 'external'}
        assert not stat.S_ISLNK(item.external_attr >> 16) and not item.flag_bits & 1
        target = (WORKSPACE / relative).resolve()
        assert target.is_relative_to(WORKSPACE.resolve())
        data = archive.read(item)
        expected = record['files_sha256'].get(item.filename, hashlib.sha256(data).hexdigest())
        assert hashlib.sha256(data).hexdigest() == expected
        if target.exists():
            assert target.read_bytes() == data, f'Existing file differs: {relative}'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
print({key: record[key] for key in ['collection_images', 'selected_images', 'requests', 'excluded', 'counts']})
print(record['limitation'])
assert record['selected_images'] == 100 and record['requests'] == 400
assert record['protocol']['question_type'] == 'candidate_Q1_Q4_clean_unscored'
assert record['protocol']['conditions'] == ['clean']
assert record['protocol']['collection'] == 'orbital_300_v4'
print('100 images / 400 Q1-Q4 inputs verified. Answers remain local, not in this ZIP.')


{'collection_images': 300, 'selected_images': 100, 'requests': 400, 'excluded': {'publisher_train_or_ineligible': 100, 'no_admitted_question': 100}, 'counts': {'mars/segmentation/test': 33, 'mars/segmentation/val': 17, 'moon/segmentation/val': 33, 'moon/segmentation/test': 17}}
Clean RGB-only Q1-Q4 CANDIDATE inference, not accepted scoring GT. Candidate answers, masks, height arrays and targets are excluded. Complete candidates do not imply final reference admission or IBM pairing. Preserve source val/test and scene groups.
100 images / 400 Q1-Q4 inputs verified. Answers remain local, not in this ZIP.


### 2. Изолированное окружение SpaceLLaVA
Ячейка скачивает Python/dependencies, но **не модель**. Kernel Colab не меняется: все команды модели запускаются отдельным Python 3.12.15. Это исключает конфликт stock Transformers/Tokenizers в Colab и не требует Rust. CUDA wheels — кандидат; следующий шаг должен проверить драйвер/GPU.
Не ставь upstream LLaVA через `pip install -e external/LLaVA`: там устаревшие Torch pins. IBM позже получит другое окружение.

In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv>=0.8,<1'], check=True)
if not (ENV_DIR / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION, '--seed', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-c', "import platform; assert platform.python_version() == '3.12.15'"], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.8.0', 'torchvision==0.23.0', '--index-url', TORCH_INDEX], check=True)
profile = WORKSPACE / 'requirements/spacellava.txt'
if RESUME:
    frozen = PERSIST / 'runs' / 'SpaceLLaVA' / RUN_ID / 'pip-freeze.txt'
    assert frozen.is_file(), 'No saved environment: use RESUME=False for a new run.'
    profile = frozen
subprocess.run([PYTHON, '-m', 'pip', 'install', '--only-binary=tokenizers', '-r', str(profile)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', str(WORKSPACE)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

def step(*arguments):
    command = [PYTHON, '-u', '-m', 'planetary_vlm.colab', *map(str, arguments), '--workspace', str(WORKSPACE)]
    logs = PERSIST / 'logs'
    logs.mkdir(exist_ok=True)
    import re
    assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,90}', RUN_ID)
    log_path = logs / f'{RUN_ID}_{SESSION_ID[:8]}_{arguments[0]}.log'
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        with log_path.open('a', encoding='utf-8') as log:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
        code = process.wait()
        if code: raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait(timeout=10)
        raise
    finally:
        process.stdout.close()

step('verify')
step('environment')

{
  "selected_images": 100,
  "requests": 400,
  "excluded": {
    "publisher_train_or_ineligible": 100,
    "no_admitted_question": 100
  },
  "counts": {
    "mars/segmentation/test": 33,
    "mars/segmentation/val": 17,
    "moon/segmentation/val": 33,
    "moon/segmentation/test": 17
  },
  "bundle_sha256": "acf9802e6d6b387bdf4c1a5752163eded5f3496d742e6064f8f59bc48f763185"
}
{
  "dependencies": {
    "python": "3.12.15",
    "planetary-vlm-benchmark": "0.2.0",
    "Pillow": "12.3.0",
    "numpy": "2.2.6",
    "torch": "2.8.0+cu128",
    "transformers": "4.37.2",
    "accelerate": "0.27.2",
    "bitsandbytes": "0.48.2",
    "peft": "0.9.0",
    "terratorch": null,
    "ni_lfm": null,
    "llava": null,
    "opencv-python-headless": "4.13.0.92",
    "PyYAML": "6.0.3",
    "huggingface-hub": "0.34.4",
    "tokenizers": "0.15.2",
    "safetensors": "0.8.0",
    "sentencepiece": "0.2.2",
    "timm": "0.6.13",
    "einops": "0.6.1"
  },
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "gpu_memor

## Steps
### 3. Локальные веса и pinned LLaVA
Для первой загрузки включи `DOWNLOAD_WEIGHTS=True` в параметрах и повтори эту ячейку. Веса публичные; helper не скачивает train-датасеты. Требуются не менее 35 GiB свободного локального диска; реальный размер пакетов/весов может меняться.
Если веса уже есть на Drive, задай `WEIGHTS_SOURCE` на папку с `SpaceLLaVA/` и `clip-vit-large-patch14-336/`. Копирование только на локальный диск, без изменения кеша. Git repo подключается по точному commit. HF_TOKEN, если нужен, передай через Colab Secrets/environment, не записывай в notebook/логи. Исходный SSL не отключать.

In [4]:
PREVIOUS_WORKSPACE = Path('/content/planetary_colab')
for name in ('weights', 'external'):
    source, target = PREVIOUS_WORKSPACE / name, WORKSPACE / name
    if source.is_dir() and not target.exists():
        target.symlink_to(source, target_is_directory=True)
        print('Reusing without copying:', target)


In [5]:
if WEIGHTS_SOURCE is not None:
    cache = Path(WEIGHTS_SOURCE)
    for name in ['SpaceLLaVA', 'clip-vit-large-patch14-336']:
        destination = WORKSPACE / 'weights' / name
        if not destination.exists():
            shutil.copytree(cache / name, destination)
arguments = ['assets', '--clone-runtime']  # Small pinned code clone, even when using existing weights.
if DOWNLOAD_WEIGHTS: arguments.append('--download')
step(*arguments)

Cloning into '/content/planetary_q1_q4_v5/external/LLaVA'...
HEAD is now at c121f04 Update README.md

Fetching 15 files: 100%|██████████| 15/15 [04:50<00:00, 19.35s/it]

Fetching 3 files: 100%|██████████| 3/3 [00:13<00:00,  4.37s/it]
{
  "assets_available": true,
  "gpu_verified": false
}


### 4. Обязательный smoke test: один clean val-запрос

Реальная загрузка модели, генерация и проверка формата закрытого ответа. Новый ZIP требует нового smoke. OOM, CPU/disk-offload, пустой ответ или неподходящий формат блокируют batch. Не заменяй результат mock-ответом. Успех smoke не означает правильный ответ и не гарантирует формат на всех четырёх задачах. Просмотри сырой ответ и затем включи batch осознанно.

In [10]:
step('smoke', '--persist', PERSIST, '--run-id', RUN_ID, '--session-id', SESSION_ID, '--checkpoint-every', 1)
proof = json.loads((WORKSPACE / 'colab_smoke_passed.json').read_text())
assert proof['success'] is True and proof['session_id'] == SESSION_ID
print('GPU/model execution admitted for this session. No scientific score was computed.')

{"run_id": "spacellava_q1_q4_v5_clean_smoke_b0907cb5", "backend": "spacellava", "mock": false, "component": "colab.runner", "pid": 13439, "total": 1, "event": "smoke_started", "level": "info", "timestamp": "2026-10-15T20:26:12.158461Z"}
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/content/venv_spacellava/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/content/venv_spacellava/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta para

### 5. Последовательный прогон 400 запросов

После успешного smoke включи `RUN_BENCHMARK=True` в параметрах и повтори ячейку запуска. Не перезапускай параметры посередине той же сессии: там создаётся новый SESSION_ID; тогда сначала нужен новый smoke. Удобнее отдельной ячейкой выполнить `RUN_BENCHMARK = True`.

Четыре вопроса × 100 снимков = 400 запросов, только clean, batch size 1. Эталоны остаются локально и не передаются модели. Checkpoint на Drive каждые 10 ответов; жёсткий disconnect может потерять последние не перенесённые ответы. Не меняй inputs/config/RUN_ID для обхода resume.

In [11]:
if RUN_BENCHMARK:
    arguments = ['run', '--persist', PERSIST, '--run-id', RUN_ID, '--session-id', SESSION_ID, '--checkpoint-every', CHECKPOINT_EVERY]
    if RESUME: arguments.append('--resume')
    if MAX_REQUESTS is not None: arguments += ['--max-requests', MAX_REQUESTS]
    step(*arguments)
else:
    print('Large run disabled. Set RUN_BENCHMARK=True after the successful smoke test.')

{"run_id": "spacellava_q1_q4_v5_clean", "backend": "spacellava", "mock": false, "component": "colab.runner", "pid": 14703, "total": 400, "event": "benchmark_started", "level": "info", "timestamp": "2026-10-15T20:31:11.276644Z"}
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/content/venv_spacellava/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/content/venv_spacellava/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in 

## Checks

Ответы и технические метаданные сохраняются в `PERSIST/runs/SpaceLLaVA/RUN_ID`. Проверяй 400 уникальных request_id и статусы. `ok` означает выполнение, не правильность. Четыре ответа на один снимок зависимы; сцены/источники также могут быть связаны. Accuracy допустима только после отдельного допуска локальных эталонов; IBM pairing ещё не выполнен.

In [ ]:
result_dir = PERSIST / 'runs' / 'SpaceLLaVA' / RUN_ID
predictions = result_dir / 'predictions.jsonl'
if predictions.exists():
    from collections import Counter
    rows = [json.loads(line) for line in predictions.read_text().splitlines() if line.strip()]
    print('Saved responses:', len(rows), 'Runtime status:', dict(Counter(row['status'] for row in rows)))
    print('Example:', [{key: row[key] for key in ['request_id', 'status', 'raw_response']} for row in rows[:2]])
else:
    print('No benchmark checkpoint yet; smoke results have a separate run ID.')

## Next Steps
После disconnect: снова GPU → Drive → тот же ZIP/WORKSPACE/RUN_ID → installation → восстановить pinned assets → новый smoke → `RESUME=True`, `RUN_BENCHMARK=True`. Если checkpoint не существует, запускай с RESUME=False. Resume откажется при изменении inputs/config/dependencies/GPU type. UUID сессии меняется, но вопросы/seed/условия не меняются.
IBM запускается позже **в отдельном окружении**, с теми же байтами допустимых входов. Публичная IMP-head не поддерживает все 200 изображений/задач: сопоставимую подвыборку необходимо выбрать отдельно. Не называть этот Colab-ноутбук сравнением encoder/heads или готовым научным benchmark.
Источники: [Colab limits](https://research.google.com/colaboratory/faq.html), [uv Python management](https://docs.astral.sh/uv/guides/install-python/), [PyTorch 2.8 wheels](https://pytorch.org/get-started/previous-versions/#v280), [SpaceLLaVA](https://huggingface.co/remyxai/SpaceLLaVA), [pinned LLaVA](https://github.com/haotian-liu/LLaVA/tree/c121f0432da27facab705978f83c4ada465e46fd).
**Execution gap:** требуется фактическое выполнение всех ячеек на GPU Colab: CUDA operation → assets preflight → fresh one-image smoke → batch run → disconnect/resume. Локальные CPU fixtures не подтверждают эти GPU/FUSE проверки.